In [ ]:
from textacy import preprocessing
import trafilatura
from bs4 import BeautifulSoup as bs
import pandas as pd

In [ ]:
df = pd.DataFrame(columns = ["Country", "Nome_Viaggio", "Link", "Descrizione Viaggio", "Mood Viaggio", "Itinerario"])

url = "https://www.sivola.it"
downloaded = trafilatura.fetch_url(url + "/viaggi")
soup = bs(downloaded, 'lxml')
l = soup.find_all(attrs={"class": "card-viaggio rounded"})
set_urls = set()
for i, el in enumerate(l):
    print(i)
    link_element = el.find(attrs={"class": "font-weight-bolder h4"}).find('a')
    nome_viaggio = link_element.text.strip()
    url_viaggio = url + link_element.get('href')
    if url_viaggio in set_urls:
        continue
    else:
        set_urls.add(url_viaggio)
    
    downloaded_viaggio = trafilatura.fetch_url(url_viaggio)
    soup_viaggio = bs(downloaded_viaggio, 'lxml')
    
    descr_viaggio = soup_viaggio.find(attrs = {"class": "h3 viaggio--title"}).text.strip()
    prose_viaggio = soup_viaggio.find(attrs = {"class": "prose"}).text.strip()
    if "_________________________________________________________________" in prose_viaggio:
        prose_viaggio = prose_viaggio.split("_________________________________________________________________")[1].strip()
    
    descr_viaggio = descr_viaggio + "\n" + prose_viaggio
    
    card_elements = soup_viaggio.find_all(attrs = {"class": "card-ranking"})
    mood_viaggio = {}
    for card in card_elements:
        card_type = card.find("small").text
        card_score = card.find("p").text
        mood_viaggio[card_type] = card_score
    
    itinerario_dict = {}
    itinerario = soup_viaggio.find(attrs = {"class": "row mt-5 mb-5"})
    tappe_itinerario = itinerario.find_all(attrs = {"class": "card-home card-tappa rounded p-4"})
    for day, tappa in enumerate(tappe_itinerario):
        itinerario_dict[f"Giorno {day+1}"] = {}
        titolo_giorno = tappa.find(attrs = {"class": "h4 font-weight-bolder pr-1"}).text.strip()
        sottotitolo_giorno = tappa.find(attrs = {"class": "mb-0 color-h text-small"}).text.strip()
        testo_giorno = tappa.find(attrs = {"class": "mt-4"}).text.strip()
        itinerario_dict[f"Giorno {day+1}"]['Title'] = titolo_giorno
        itinerario_dict[f"Giorno {day+1}"]['Subtitle'] = sottotitolo_giorno
        itinerario_dict[f"Giorno {day+1}"]['Tappe'] = testo_giorno
        
    row = pd.DataFrame.from_dict({"Country": [nome_viaggio.split(":")[0].strip()],
                             "Nome_Viaggio": [nome_viaggio],
                              "Link": [url_viaggio],
                              "Descrizione Viaggio": [descr_viaggio],
                              "Mood Viaggio": [mood_viaggio],
                             "Itinerario": [itinerario_dict]})
    df = pd.concat([df, row], ignore_index=True)
    print("*"*80)

In [ ]:
df

In [ ]:
df.to_pickle("df_viaggi_sivola.pkl")